# Arabic LLM Quantisation

Reproduce the FP16, custom GPTQ 4-bit and custom GPTQ 8-bit comparison for Qwen2.5-1.5B-Instruct.

Select a T4 GPU in Colab and run the cells in order. Each quality run evaluates 30 ArabicMMLU test questions.

## 1. Check the GPU and install the tools

In [1]:
!nvidia-smi

Thu Nov 26 20:28:03 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   49C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
%%bash
set -e
python -m pip install -q uv
python -m uv venv /content/qwen-vllm
python -m uv pip install --python /content/qwen-vllm/bin/python vllm==0.34.0 --torch-backend=auto
/content/qwen-vllm/bin/vllm --version

0.34.0


Using CPython 3.13.16 interpreter at: /usr/bin/python3
Creating virtual environment at: qwen-vllm
Activate with: source qwen-vllm/bin/activate
Using Python 3.13.16 environment at: qwen-vllm
Resolved 199 packages in 1.60s
Prepared 199 packages in 1m 49s
Installed 199 packages in 1.44s
 + agent-detector==2.0.0
 + aiohappyeyeballs==2.7.1
 + aiohttp==3.14.5
 + aiosignal==1.4.0
 + annotated-doc==0.0.7
 + annotated-types==0.8.0
 + anthropic==2.14.0
 + anyio==4.16.0
 + apache-tvm-ffi==0.1.11
 + astor==0.8.1
 + attrs==26.1.0
 + blake3==1.0.11
 + cachetools==8.0.3
 + cbor2==6.3.0
 + certifi==2026.11.20
 + cffi==2.1.1
 + charset-normalizer==3.5.3
 + click==8.5.1
 + cloudpickle==3.1.3
 + compressed-tensors==0.17.0
 + cryptography==50.0.2
 + cuda-bindings==13.5.0
 + cuda-core==1.2.2
 + cuda-pathfinder==1.9.1
 + cuda-python==13.5.0
 + cuda-tile==1.8.1
 + cuda-toolkit==13.2.1
 + depyf==0.20.0
 + detect-installer==0.2.1
 + dill==0.4.1
 + dnspython==2.9.2
 + docstring-parser==0.18.0
 + einops==0.8.2
 

In [3]:
%%bash
set -e
python -m uv venv /content/qwen-quant
python -m uv pip install --python /content/qwen-quant/bin/python \
    gptqmodel==7.12.2 datasets==5.2.0 --torch-backend=auto
/content/qwen-quant/bin/python -c 'import torch; from gptqmodel import GPTQModel, QuantizeConfig; print("GPTQModel installed"); print("CUDA available:", torch.cuda.is_available())'


INFO  LogBar: headless/CI mode; progress animations disabled.                   
WARN  Python GIL is enabled: Multi-gpu quant acceleration for MoE models is sub-optimal and multi-core accelerated cpu packing is also disabled. We recommend Python >= 3.13.3t with Pytorch > 2.8 for mult-gpu quantization and multi-cpu packing with env `PYTHON_GIL=0`.
INFO  ENV: Auto setting PYTORCH_ALLOC_CONF='expandable_segments:True' for memory saving.
INFO  ENV: Auto setting CUDA_DEVICE_ORDER=PCI_BUS_ID for correctness.           
INFO    ┌─────────────┐    ┌────────────────────────┐    ┌────────────┐    ┌─────────┐ │ GPT-QModel  │ -> │ ▓▓▓▓▓▓▓▓▓▓▓▓ 16bit     │ -> │ ▒▒▒▒ 8bit  │ -> │ ░░ 4bit │ └─────────────┘    └────────────────────────┘    └────────────┘    └─────────┘ GPT-QModel   : 7.12.2 Transformers : 5.24.1 Torch        : 2.15.0+cu132 Triton       : 3.9.0
GPTQModel installed
CUDA available: True


Using CPython 3.13.16 interpreter at: /usr/bin/python3
Creating virtual environment at: qwen-quant
Activate with: source qwen-quant/bin/activate
Using Python 3.13.16 environment at: qwen-quant
Resolved 83 packages in 1.39s
Prepared 32 packages in 1m 16s
Installed 83 packages in 820ms
 + accelerate==1.16.0
 + annotated-doc==0.0.7
 + annotated-types==0.8.0
 + anyio==4.16.0
 + certifi==2026.11.20
 + charset-normalizer==3.5.3
 + click==8.5.1
 + cuda-bindings==13.5.0
 + cuda-pathfinder==1.9.1
 + cuda-toolkit==13.2.2
 + datasets==5.2.0
 + defuser==0.0.33
 + device-smi==0.5.7
 + dill==0.4.1
 + filelock==4.3.1
 + fsspec==2026.9.0
 + gptqmodel==7.12.2
 + h11==0.16.0
 + hf-xet==1.8.0
 + httpcore2==2.16.2
 + httpx2==2.16.2
 + huggingface-hub==2.5.0
 + idna==3.20
 + jinja2==3.1.6
 + logbar==0.4.16
 + markdown-it-py==4.3.0
 + markupsafe==3.0.4
 + mdurl==0.1.2
 + mpmath==1.3.0
 + multiprocess==0.70.19
 + networkx==3.7
 + ninja==1.13.2
 + numpy==2.5.5
 + nvidia-cublas==13.4.1.3
 + nvidia-cuda-cccl==1

## 2. Define the benchmarks and measure FP16

In [4]:
import os
import signal
import subprocess
import time
from pathlib import Path

import requests

MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
SERVER_URL = "http://127.0.0.1:8000"

def benchmark_speed():
    import subprocess
    import time
    from statistics import median

    import requests

    url = "http://127.0.0.1:8000/v1/chat/completions"
    prompts = [
        "اشرح فوائد القراءة في ثلاث جمل.",
        "اشرح كيف يتكوّن المطر في ثلاث جمل.",
        "اذكر ثلاث طرق لتوفير الماء في المنزل واشرحها.",
        "اشرح الفرق بين الطقس والمناخ مع مثال.",
        "اكتب فقرة قصيرة عن أهمية الرياضة.",
    ]

    def ask(prompt):
        start = time.perf_counter()
        response = requests.post(
            url,
            json={
                "model": MODEL,
                "messages": [{"role": "user", "content": prompt}],
                "temperature": 0,
                "max_tokens": 64,
            },
            timeout=90,
        )
        response.raise_for_status()
        elapsed = time.perf_counter() - start
        return elapsed, response.json()["usage"]["completion_tokens"]

    ask("اكتب جملتين عن أهمية النوم.")  # إحماء

    samples = [ask(prompt) for prompt in prompts]
    latencies, token_counts = zip(*samples)

    gpu_memory = subprocess.check_output(
        ["nvidia-smi", "-i", "0", "--query-gpu=memory.used",
         "--format=csv,noheader,nounits"],
        text=True,
    ).strip()

    print(f"Median response time: {median(latencies):.2f} s")
    print(f"Output throughput: {sum(token_counts) / sum(latencies):.2f} tokens/s")
    print(f"GPU memory with server running: {gpu_memory} MiB")

def benchmark_accuracy():
    import csv
    import io
    import random

    import requests

    MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
    API_URL = "http://127.0.0.1:8000/v1/chat/completions"
    DATA_URL = (
        "https://huggingface.co/datasets/MBZUAI/ArabicMMLU/"
        "resolve/9ea4eb3/All/test.csv"
    )
    TARGET = 30

    data = requests.get(DATA_URL, timeout=60)
    data.raise_for_status()

    rows = list(csv.DictReader(io.StringIO(data.content.decode("utf-8-sig"))))
    candidates = random.Random(42).sample(rows, 100)

    completed = 0
    correct = 0
    unformatted = 0
    skipped = 0

    for row in candidates:
        options = []
        valid_labels = set()

        for index, letter in enumerate("ABCDE", start=1):
            option = (row.get(f"Option {index}") or "").strip()
            if option:
                options.append(f"{letter}. {option}")
                valid_labels.add(letter)

        parts = [
            "اختر الإجابة الصحيحة. اكتب حرف الخيار فقط من الخيارات المعروضة.",
            (row.get("Context") or "").strip(),
            row["Question"],
            *options,
        ]
        prompt = "\n".join(part for part in parts if part)

        response = requests.post(
            API_URL,
            json={
                "model": MODEL,
                "messages": [{"role": "user", "content": prompt}],
                "temperature": 0,
                "max_tokens": 8,
            },
            timeout=90,
        )

        if (
            response.status_code == 400
            and "maximum context length" in response.text.lower()
        ):
            skipped += 1
            continue

        response.raise_for_status()

        reply = response.json()["choices"][0]["message"]["content"] or ""
        answer = reply.strip().strip("*.() ").upper()
        expected = row["Answer Key"].strip().upper()

        completed += 1
        correct += answer == expected
        unformatted += answer not in valid_labels

        if completed % 10 == 0:
            print(f"تم اختبار {completed}/{TARGET}", flush=True)
        if completed == TARGET:
            break

    if completed != TARGET:
        raise RuntimeError(f"اكتمل {completed} سؤالًا فقط من أصل {TARGET}.")

    print(f"Skipped due to context length: {skipped}")
    print(f"Accuracy: {correct}/{completed} ({correct / completed:.1%})")
    print(f"Replies without a single valid option letter: {unformatted}")

def evaluate_checkpoint(label, checkpoint, log_path):
    print(f"Evaluating {label}", flush=True)
    with open(log_path, "w") as log:
        process = subprocess.Popen(
            [
                "/content/qwen-vllm/bin/vllm", "serve", checkpoint,
                "--served-model-name", MODEL,
                "--dtype", "half",
                "--max-model-len", "2048",
                "--enforce-eager",
                "--host", "127.0.0.1",
                "--port", "8000",
            ],
            stdout=log,
            stderr=subprocess.STDOUT,
            start_new_session=True,
        )

    try:
        deadline = time.monotonic() + 600
        while time.monotonic() < deadline:
            if process.poll() is not None:
                raise RuntimeError(f"vLLM exited with code {process.returncode}.")
            try:
                response = requests.get(f"{SERVER_URL}/v1/models", timeout=2)
                if response.ok and response.json()["data"][0]["root"] == checkpoint:
                    break
            except requests.RequestException:
                pass
            time.sleep(2)
        else:
            raise TimeoutError(f"vLLM did not start within 10 minutes: {checkpoint}")

        benchmark_speed()
        benchmark_accuracy()

        for line in Path(log_path).read_text().splitlines():
            if "Model loading took" in line:
                print(line)

    except Exception:
        print("\n".join(Path(log_path).read_text().splitlines()[-20:]))
        raise

    finally:
        try:
            os.killpg(process.pid, signal.SIGTERM)
        except ProcessLookupError:
            pass
        process.wait(timeout=60)

        for _ in range(60):
            used_mib = subprocess.check_output(
                ["nvidia-smi", "-i", "0", "--query-gpu=memory.used",
                 "--format=csv,noheader,nounits"],
                text=True,
            )
            if int(used_mib.strip()) < 1024:
                break
            time.sleep(1)
        else:
            raise RuntimeError("GPU memory was not released within 60 seconds.")

In [5]:
evaluate_checkpoint("FP16", MODEL, "/content/qwen-fp16.log")

Evaluating FP16
Median response time: 2.17 s
Output throughput: 28.26 tokens/s
GPU memory with server running: 13435 MiB
تم اختبار 10/30
تم اختبار 20/30
تم اختبار 30/30
Skipped due to context length: 2
Accuracy: 12/30 (40.0%)
Replies without a single valid option letter: 7
(EngineCore pid=3240) INFO 11-26 20:36:33 [model_runner.py:441] Model loading took 2.98 GiB memory and 42.011561 seconds


## 3. Create and evaluate the custom 4-bit checkpoint

In [6]:
%%bash
set -e

/content/qwen-quant/bin/python -u - <<'PY'
import csv
import io
import random

import requests
from gptqmodel import CalibrationConfig, GPTQConfig, GPTQModel
from gptqmodel.backend import RuntimeConfig

BITS = 4
BASE_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
OUTPUT = f"/content/qwen-custom-gptq-{BITS}bit"
DEV_URL = (
    "https://huggingface.co/datasets/MBZUAI/ArabicMMLU/"
    "resolve/9ea4eb3/All/dev.csv"
)

response = requests.get(DEV_URL, timeout=60)
response.raise_for_status()
rows = list(csv.DictReader(io.StringIO(response.content.decode("utf-8-sig"))))

calibration = []
for row in random.Random(42).sample(rows, 64):
    options = []
    for index, letter in enumerate("ABCDE", start=1):
        option = (row.get(f"Option {index}") or "").strip()
        if option:
            options.append(f"{letter}. {option}")

    parts = [
        "اختر الإجابة الصحيحة.",
        (row.get("Context") or "").strip(),
        row["Question"],
        *options,
        f"الإجابة: {row['Answer Key']}",
    ]
    calibration.append("\n".join(part for part in parts if part))

print(f"Starting {BITS}-bit GPTQ with {len(calibration)} calibration samples.")

model = GPTQModel.load(
    BASE_MODEL,
    GPTQConfig(
        bits=BITS,
        group_size=128,
        format="gptq",
        runtime=RuntimeConfig(cpu_workers=2),
    ),
)

model.quantize(CalibrationConfig(dataset=calibration, batch_size=1))
model.save(OUTPUT)
print(f"Saved: {OUTPUT}")
PY


INFO  LogBar: headless/CI mode; progress animations disabled.                   
WARN  Python GIL is enabled: Multi-gpu quant acceleration for MoE models is sub-optimal and multi-core accelerated cpu packing is also disabled. We recommend Python >= 3.13.3t with Pytorch > 2.8 for mult-gpu quantization and multi-cpu packing with env `PYTHON_GIL=0`.
INFO  ENV: Auto setting PYTORCH_ALLOC_CONF='expandable_segments:True' for memory saving.
INFO  ENV: Auto setting CUDA_DEVICE_ORDER=PCI_BUS_ID for correctness.           
INFO    ┌─────────────┐    ┌────────────────────────┐    ┌────────────┐    ┌─────────┐ │ GPT-QModel  │ -> │ ▓▓▓▓▓▓▓▓▓▓▓▓ 16bit     │ -> │ ▒▒▒▒ 8bit  │ -> │ ░░ 4bit │ └─────────────┘    └────────────────────────┘    └────────────┘    └─────────┘ GPT-QModel   : 7.12.2 Transformers : 5.24.1 Torch        : 2.15.0+cu132 Triton       : 3.9.0
Starting 4-bit GPTQ with 64 calibration samples.
INFO  QuantizeConfig: offload_to_disk_path auto set to temporary dir `/tmp/gptqmodel_6m3vtia6

W1126 20:38:37.779000 4148 torch/utils/_pytree.py:646] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W1126 20:38:37.820000 4148 torch/utils/_pytree.py:646] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
INFO:httpx2:HTTP Request: GET https://huggingface.co/api/models/Qwen/Qwen2.5-1.5B-Instruct/tree/main?recursive=true&expand=false "HTTP/1.1 200 OK"
INFO:httpx2:HTTP Request: GET https://huggingface.co/api/models/Qwen/Qwen2.5-1.5B-Instruct "HTTP/1.1 200 OK"
INFO:httpx2:HTTP Request: GET https://huggingface.co/api/models/Qwen/Qwen2.5-1.5B-Instruct/tree/main?recursive=true&expand=false "HTTP/1.1 200 OK"
INFO:httpx2:HTTP Request: GET https://hug

In [7]:
evaluate_checkpoint("Custom GPTQ 4-bit", "/content/qwen-custom-gptq-4bit", "/content/qwen-custom-4bit.log")

Evaluating Custom GPTQ 4-bit
Median response time: 2.86 s
Output throughput: 22.04 tokens/s
GPU memory with server running: 13403 MiB
تم اختبار 10/30
تم اختبار 20/30
تم اختبار 30/30
Skipped due to context length: 2
Accuracy: 3/30 (10.0%)
Replies without a single valid option letter: 16
(EngineCore pid=6744) INFO 11-26 20:48:43 [model_runner.py:441] Model loading took 1.1 GiB memory and 8.616612 seconds


## 4. Create and evaluate the custom 8-bit checkpoint

In [8]:
%%bash
set -e

/content/qwen-quant/bin/python -u - <<'PY'
import csv
import io
import random

import requests
from gptqmodel import CalibrationConfig, GPTQConfig, GPTQModel
from gptqmodel.backend import RuntimeConfig

BITS = 8
BASE_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
OUTPUT = f"/content/qwen-custom-gptq-{BITS}bit"
DEV_URL = (
    "https://huggingface.co/datasets/MBZUAI/ArabicMMLU/"
    "resolve/9ea4eb3/All/dev.csv"
)

response = requests.get(DEV_URL, timeout=60)
response.raise_for_status()
rows = list(csv.DictReader(io.StringIO(response.content.decode("utf-8-sig"))))

calibration = []
for row in random.Random(42).sample(rows, 64):
    options = []
    for index, letter in enumerate("ABCDE", start=1):
        option = (row.get(f"Option {index}") or "").strip()
        if option:
            options.append(f"{letter}. {option}")

    parts = [
        "اختر الإجابة الصحيحة.",
        (row.get("Context") or "").strip(),
        row["Question"],
        *options,
        f"الإجابة: {row['Answer Key']}",
    ]
    calibration.append("\n".join(part for part in parts if part))

print(f"Starting {BITS}-bit GPTQ with {len(calibration)} calibration samples.")

model = GPTQModel.load(
    BASE_MODEL,
    GPTQConfig(
        bits=BITS,
        group_size=128,
        format="gptq",
        runtime=RuntimeConfig(cpu_workers=2),
    ),
)

model.quantize(CalibrationConfig(dataset=calibration, batch_size=1))
model.save(OUTPUT)
print(f"Saved: {OUTPUT}")
PY


INFO  LogBar: headless/CI mode; progress animations disabled.                   
WARN  Python GIL is enabled: Multi-gpu quant acceleration for MoE models is sub-optimal and multi-core accelerated cpu packing is also disabled. We recommend Python >= 3.13.3t with Pytorch > 2.8 for mult-gpu quantization and multi-cpu packing with env `PYTHON_GIL=0`.
INFO  ENV: Auto setting PYTORCH_ALLOC_CONF='expandable_segments:True' for memory saving.
INFO  ENV: Auto setting CUDA_DEVICE_ORDER=PCI_BUS_ID for correctness.           
INFO    ┌─────────────┐    ┌────────────────────────┐    ┌────────────┐    ┌─────────┐ │ GPT-QModel  │ -> │ ▓▓▓▓▓▓▓▓▓▓▓▓ 16bit     │ -> │ ▒▒▒▒ 8bit  │ -> │ ░░ 4bit │ └─────────────┘    └────────────────────────┘    └────────────┘    └─────────┘ GPT-QModel   : 7.12.2 Transformers : 5.24.1 Torch        : 2.15.0+cu132 Triton       : 3.9.0
Starting 8-bit GPTQ with 64 calibration samples.
INFO  QuantizeConfig: offload_to_disk_path auto set to temporary dir `/tmp/gptqmodel_vt4wrixa

W1126 20:50:29.158000 7322 torch/utils/_pytree.py:646] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W1126 20:50:29.211000 7322 torch/utils/_pytree.py:646] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
INFO:httpx2:HTTP Request: GET https://huggingface.co/api/models/Qwen/Qwen2.5-1.5B-Instruct/tree/main?recursive=true&expand=false "HTTP/1.1 200 OK"
INFO:httpx2:HTTP Request: GET https://huggingface.co/api/models/Qwen/Qwen2.5-1.5B-Instruct "HTTP/1.1 200 OK"
INFO:httpx2:HTTP Request: GET https://huggingface.co/api/models/Qwen/Qwen2.5-1.5B-Instruct/tree/main?recursive=true&expand=false "HTTP/1.1 200 OK"
INFO:httpx2:HTTP Request: GET https://hug

In [9]:
evaluate_checkpoint("Custom GPTQ 8-bit", "/content/qwen-custom-gptq-8bit", "/content/qwen-custom-8bit.log")

Evaluating Custom GPTQ 8-bit
Median response time: 2.88 s
Output throughput: 21.44 tokens/s
GPU memory with server running: 13429 MiB
تم اختبار 10/30
تم اختبار 20/30
تم اختبار 30/30
Skipped due to context length: 2
Accuracy: 12/30 (40.0%)
Replies without a single valid option letter: 7
(EngineCore pid=9110) INFO 11-26 20:57:20 [model_runner.py:441] Model loading took 1.74 GiB memory and 10.188165 seconds
